In [1]:
import pandas as pd
import ast
import os

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
for genus_name in keep_genus:
    print(genus_name)
    replicon_info = pd.read_csv(f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    replicon_info = replicon_info[replicon_info['ms-label']=='NMS_replicon']
    folder = f'/active-data/analysis_results/chr_pla/genus/NMS_replicon/{genus_name}/'
    data_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/annotations'
    os.makedirs(data_dir, exist_ok=True)
    
    os.chdir(folder)
    %time os.system(f'amrfinder -n NMS_replicon_nucleotide_seq.fasta -o {data_dir}/amrfinder_NMS_replicon_result.txt --threads 16 --quiet')
    
    amr_resu = pd.read_csv(f'{data_dir}/amrfinder_NMS_replicon_result.txt', sep='\t')
    amr_resu = amr_resu[amr_resu['Type'] == 'AMR']
    
    amr_resu.rename(columns={'Contig id': 'accession', 'Type': 'AMR_type'}, inplace=True)
    replicon_type = pd.merge(replicon_info[['accession']], amr_resu[['accession', 'AMR_type']], on='accession', how='left').fillna('non-AMR')
    replicon_type.to_csv(f'{data_dir}/NMS_replicon_AMRtyper_results.tsv', sep = '\t',)

Escherichia
CPU times: user 93 ms, sys: 37.5 ms, total: 131 ms
Wall time: 22min 5s
Klebsiella
CPU times: user 175 ms, sys: 98.1 ms, total: 273 ms
Wall time: 48min 49s
Staphylococcus
CPU times: user 2.05 ms, sys: 2.06 ms, total: 4.11 ms
Wall time: 26.7 s
Pseudomonas
CPU times: user 12.4 ms, sys: 5.35 ms, total: 17.7 ms
Wall time: 2min 32s
Bacillus
CPU times: user 7.81 ms, sys: 7.48 ms, total: 15.3 ms
Wall time: 2min 47s
Salmonella
CPU times: user 26.2 ms, sys: 11 ms, total: 37.2 ms
Wall time: 7min 9s
Streptococcus
CPU times: user 1.48 ms, sys: 44 μs, total: 1.53 ms
Wall time: 1.72 s
Streptomyces
CPU times: user 7.69 ms, sys: 2.31 ms, total: 10 ms
Wall time: 1min 45s
Acinetobacter
CPU times: user 7.93 ms, sys: 7.3 ms, total: 15.2 ms
Wall time: 2min 6s
Enterococcus
CPU times: user 6.68 ms, sys: 2.32 ms, total: 9 ms
Wall time: 1min 15s
Bordetella
CPU times: user 1.47 ms, sys: 42 μs, total: 1.51 ms
Wall time: 884 ms
Enterobacter
CPU times: user 29.6 ms, sys: 15.3 ms, total: 44.9 ms
Wall tim